In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# ----------------------------
# Paths (DO NOT CHANGE THESE)
# ----------------------------
TEST_CSV = '/kaggle/input/csiro-biomass/test.csv'
TEST_IMG_DIR = '/kaggle/input/csiro-biomass/test' 
WEIGHTS_DIR = '/kaggle/input/csiro-resnet-18-version-2/'  # ← YOUR DATASET NAME

# ----------------------------
# Load normalization stats
# ----------------------------
outer_mean = np.load(WEIGHTS_DIR + 'outer_mean.npy')
outer_std = np.load(WEIGHTS_DIR + 'outer_std.npy')
target_columns = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']

# ----------------------------
# Model definition (MUST match training)
# ----------------------------
def create_model_RN18():
    model = models.resnet18(weights=None)  # no internet needed
    model.fc = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 5)
    )
    return model

# ----------------------------
# Dataset
# ----------------------------
class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        img = Image.open(os.path.join(self.img_dir, img_path)).convert('RGB')
        return self.transform(img), img_path

# ----------------------------
# Transforms (300x300 for B3)
# ----------------------------
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),        
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# ----------------------------
# Load test data
# ----------------------------
df_test = pd.read_csv(TEST_CSV)
df_test['image_path'] = df_test['image_path'].str.replace(r'^test/', '', regex=True)
df_unique = df_test[['image_path']].drop_duplicates().reset_index(drop=True)
print(f"Predicting on {len(df_unique)} images")

# ----------------------------
# Load ensemble models
# ----------------------------
model_paths = [f"{WEIGHTS_DIR}best_model_fold_ResNet18_3cv_{i}.pth" for i in range(3)]
models_list = []

for path in model_paths:
    model = create_model_RN18()
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device).eval()
    models_list.append(model)

# ----------------------------
# Inference
# ----------------------------
test_ds = TestDataset(df_unique, TEST_IMG_DIR, eval_transform)
test_dl = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

all_img_paths = []
all_preds = []

with torch.no_grad():
    for imgs, paths in test_dl:
        preds = torch.stack([m(imgs.to(device)) for m in models_list]).mean(0)
        all_preds.append(preds.cpu())
        all_img_paths.extend(paths)

preds = torch.cat(all_preds, dim=0).numpy()
preds_orig = preds * outer_std + outer_mean  # denormalize

# ----------------------------
# Format submission
# ----------------------------
rows = []
for i, img_path in enumerate(all_img_paths):
    img_id = os.path.splitext(os.path.basename(img_path))[0]  # e.g., "ID1001187975"
    for j, target in enumerate(target_columns):
        sample_id = f"{img_id}__{target}"
        rows.append({"sample_id": sample_id, "target": preds_orig[i, j]})

submission = pd.DataFrame(rows)[['sample_id', 'target']]
submission.to_csv('submission.csv', index=False)

print("✅ Submission saved!")
print(submission.head())

Device: cpu
Predicting on 1 images
✅ Submission saved!
                    sample_id     target
0  ID1001187975__Dry_Clover_g   2.963288
1    ID1001187975__Dry_Dead_g  25.463426
2   ID1001187975__Dry_Green_g  25.086802
3   ID1001187975__Dry_Total_g  53.422808
4         ID1001187975__GDM_g  28.256405
